In [1]:
!pip install fastapi uvicorn librosa soundfile numpy requests pydub

In [2]:
!pip install pyngrok

In [3]:

from fastapi import FastAPI, Form, HTTPException
from fastapi.responses import FileResponse, JSONResponse
import librosa
import soundfile as sf
import numpy as np
import requests
import tempfile
import os
import json
import time
from typing import List
import logging
from typing import Tuple

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="VibeShift Colab API", version="1.0.0")

In [4]:
def _normalize_audio(y: np.ndarray):
    """Normalize to peak."""
    peak = np.max(np.abs(y))
    if peak > 0:
        return y / peak * 0.98
    return y


def _download_stem(url: str, timeout: int = 60) -> Tuple[np.ndarray, int]:
    """Download and load stem from URL."""
    logger.info(f"Downloading stem from {url[:50]}...")
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        tmp.write(resp.content)
        tmp.flush()

        y, sr = librosa.load(tmp.name, sr=None, mono=False)

        # Reshape if needed
        if y.ndim == 2:
            if y.shape[0] <= 8 and y.shape[1] > y.shape[0]:
                y = y.T

        os.unlink(tmp.name)
        logger.info(f"Loaded stem: shape={y.shape}, sr={sr}")
        return y, sr


def _upload_to_cloud(file_path: str) -> str:
    """
    Upload processed audio to cloud storage.
    For now, return local path. In production, use GCS/S3.

    Example with GCS:
        from google.cloud import storage
        bucket = storage.Client().bucket('vibeshift-audio')
        blob = bucket.blob(os.path.basename(file_path))
        blob.upload_from_filename(file_path)
        return blob.public_url
    """
    return file_path

In [5]:
@app.get("/health")
def health():
    return {"status": "ok", "service": "VibeShift Colab API"}


@app.post("/process-stem")
async def process_stem(
    stem_url: str = Form(...),
    pitch_shift: int = Form(default=0),
    timbre_strength: float = Form(default=1.0),
    volume_db: float = Form(default=0.0),
):
    """
    Download, edit, and return a processed stem.

    Args:
        stem_url: URL of stem from Replicate
        pitch_shift: Semitones to shift (-12 to +12)
        timbre_strength: Harmonic boost (0.5 to 2.0)
        volume_db: Volume change in dB (-12 to +12)

    Returns:
        Processed stem as WAV file
    """
    try:
        logger.info(
            f"Processing stem: pitch={pitch_shift}, timbre={timbre_strength}, vol={volume_db}dB"
        )

        # Download stem
        y, sr = _download_stem(stem_url)

        # Pitch shift (GPU-optimized on Colab)
        if pitch_shift != 0:
            logger.info(f"Applying pitch shift: {pitch_shift} semitones...")
            if y.ndim == 1:
                y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch_shift)
            else:
                y = np.stack(
                    [
                        librosa.effects.pitch_shift(y[:, ch], sr=sr, n_steps=pitch_shift)
                        for ch in range(y.shape[1])
                    ],
                    axis=1,
                )

        # Timbre modification (HPSS)
        if timbre_strength != 1.0:
            logger.info(f"Applying timbre adjustment: {timbre_strength}...")
            if y.ndim == 1:
                harm, perc = librosa.effects.hpss(y)
                y = np.clip(harm * timbre_strength + perc, -1.0, 1.0)
            else:
                y_out = []
                for ch in range(y.shape[1]):
                    harm, perc = librosa.effects.hpss(y[:, ch])
                    y_ch = np.clip(harm * timbre_strength + perc, -1.0, 1.0)
                    y_out.append(y_ch)
                y = np.stack(y_out, axis=1)

        # Volume adjustment
        if volume_db != 0.0:
            logger.info(f"Applying volume adjustment: {volume_db}dB...")
            gain = 10 ** (volume_db / 20.0)
            y = np.clip(y * gain, -1.0, 1.0)

        # Normalize
        y = _normalize_audio(y)

        # Save and upload
        out_path = f"/tmp/stem_processed_{int(time.time() * 1000)}.wav"
        sf.write(out_path, y, sr)
        logger.info(f"Saved processed stem: {out_path}")

        return FileResponse(out_path, media_type="audio/wav", filename="stem.wav")

    except Exception as e:
        logger.error(f"Error processing stem: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/mix-stems")
async def mix_stems(
    stem_urls: str = Form(...),  # JSON array of URLs
):
    """
    Download multiple stems and mix them together.

    Args:
        stem_urls: JSON array of stem URLs

    Returns:
        Mixed audio as WAV file
    """
    try:
        urls = json.loads(stem_urls)
        logger.info(f"Mixing {len(urls)} stems...")

        stems = []
        sr_final = None

        for i, url in enumerate(urls):
            logger.info(f"Downloading stem {i+1}/{len(urls)}...")
            y, sr = _download_stem(url)
            stems.append(y)
            sr_final = sr

        # Find max length and pad
        max_len = max(len(s) for s in stems) if stems else 0
        mix = np.zeros(max_len, dtype=np.float32)

        # Sum stems
        for s in stems:
            if s.ndim == 1:
                mix[: len(s)] += s
            else:
                # Convert stereo to mono by averaging channels
                s_mono = s.mean(axis=1)
                mix[: len(s_mono)] += s_mono

        # Normalize
        mix = _normalize_audio(mix)

        # Save
        out_path = f"/tmp/mix_{int(time.time() * 1000)}.wav"
        sf.write(out_path, mix, sr_final)
        logger.info(f"Mixed audio saved: {out_path}")

        return FileResponse(out_path, media_type="audio/wav", filename="mixed.wav")

    except Exception as e:
        logger.error(f"Error mixing stems: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/full-pipeline")
async def full_pipeline(
    song_file: bytes = Form(...),
    pitch_shifts: str = Form(default="{}"),  # {"vocals": 2, "drums": 0}
    timbre_strengths: str = Form(default="{}"),  # {"vocals": 1.2}
    volume_adjustments: str = Form(default="{}"),
):
    """
    Full pipeline: Demix → Edit → Mix (all on Colab with GPU)

    This requires calling Replicate first from your backend.
    Use this for local testing or if you want GPU demixing too.
    """
    # For now, this is a template
    # In production, you'd call Replicate here too
    pass

In [6]:
@app.post("/process-and-mix")
async def process_and_mix(stems_json: str = Form(...)):
    try:
        stems_data = json.loads(stems_json)
        logger.info(f"process-and-mix: {len(stems_data)} stems")
        arrays = []
        sr_final = 44100
        for stem in stems_data:
            if stem.get("muted", False):
                continue
            y, sr = _download_stem(stem["url"])
            sr_final = sr
            pitch  = stem.get("pitch",  0)
            timbre = stem.get("timbre", 1.0)
            volume = stem.get("volume", 1.0)
            if pitch != 0:
                if y.ndim == 1:
                    y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
                else:
                    y = np.stack([librosa.effects.pitch_shift(y[:, ch], sr=sr, n_steps=pitch)
                                  for ch in range(y.shape[1])], axis=1)
            if timbre != 1.0:
                if y.ndim == 1:
                    harm, perc = librosa.effects.hpss(y)
                    y = np.clip(harm * timbre + perc, -1.0, 1.0)
                else:
                    y = np.stack([np.clip(librosa.effects.hpss(y[:, ch])[0] * timbre
                                          + librosa.effects.hpss(y[:, ch])[1], -1.0, 1.0)
                                  for ch in range(y.shape[1])], axis=1)
            if y.ndim == 2:
                y = y.mean(axis=1)
            arrays.append(y.astype(np.float32) * float(volume))

        if not arrays:
            raise HTTPException(status_code=400, detail="No unmuted stems to mix")

        max_len = max(len(a) for a in arrays)
        mix = np.zeros(max_len, dtype=np.float32)
        for a in arrays:
            mix[:len(a)] += a
        mix = _normalize_audio(mix)

        out_path = f"/tmp/mix_{int(time.time()*1000)}.wav"
        sf.write(out_path, mix, sr_final)
        logger.info(f"Mix done: {out_path}")
        return FileResponse(out_path, media_type="audio/wav", filename="mix.wav")

    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"process-and-mix error: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))

In [8]:
from pyngrok import ngrok
import uvicorn
import threading

# Authenticate (correct method)
ngrok.set_auth_token("3DcmWQyRNMiZZgv7Rcn20JcCu0q_3vxpAvfbo7WiWqof8gAM9")

# Kill old tunnels
ngrok.kill()

# Start tunnel
public_url = ngrok.connect(8001)
print(f"✅ Production API: {public_url}")
print(f"   Keep this notebook running 24/7")

# Start FastAPI server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Keep alive
import time
try:
    while True:
        time.sleep(60)
        print(f"✅ API alive at {public_url}")
except KeyboardInterrupt:
    print("Stopped")

✅ Production API: NgrokTunnel: "https://eating-dictate-handwrite.ngrok-free.dev" -> "http://localhost:8001"
   Keep this notebook running 24/7


INFO:     Started server process [1474]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Stopped


In [ ]:
if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=8000)


RuntimeError: asyncio.run() cannot be called from a running event loop